## Regression I: single variable regression

#### <p style="text-align: right;"> &#9989; **put your name here** </p>



---
<img src="https://imgs.xkcd.com/comics/extrapolating.png" width=400px>

In this module, we will practices several Python curve-fitting tools, including
1. `polyfit` in `numpy`.
2. `OLS` in `statsmodel`.
3. `curve_fit` in `scipy`.
4. `least_squares` in `scipy`.


<img src="https://community.cloudera.com/t5/image/serverpage/image-id/25068iFF075A5AEC3B8528/image-size/medium?v=1.0&px=300"> 

### 1. Linear function. 

Create a dataset of a linear function with noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# reproducibility
np.random.seed(0)

# x values
x = np.linspace(0, 10, 50)

# true line
y_true = 3*x + 5

# add random noise
noise = np.random.normal(loc=0, scale=2, size=len(x))

y = y_true + noise

# visualization
plt.figure(figsize=(6,4))

plt.scatter(x, y)

plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)
plt.show()

#### 1.1 `polyfit` in `numpy` library.

`np.polyfit` performs a least-squares polynomial fit. Suppose you want to fit data:

$$(x_i, y_i)$$

with a polynomial. 

**Main idea**

`polyfit` finds coefficients that minimize squared error:

$$\min \sum_i \big(y_i-p(x_i)\big)^2,$$
 
where: $p(x)$ = polynomial. Using $y = ax + b$ as an example, we shall minimize the L-2 norm of residual ($r_i = y_i - (ax_i + b)$),

$$\min \big(y_i- (ax_i+b) \big)^2$$


In matrix form,

$$\mathbf{y} = \mathbf{X} \beta,$$

where 

$$\beta = \begin{bmatrix} a \\ b \end{bmatrix}$$

and 

$$\mathbf{X} =  \begin{bmatrix} x_1 & 1 \\ x_2 & 1  \\ . \\ . \\ x_n & 1 \end{bmatrix}.$$

**Least-squares solution**

We minimize:

$$\min_\beta || \mathbf{X} \beta - \mathbf{y} ||^2 $$

where 

$$ \mathcal{F} = || \mathbf{X} \beta - \mathbf{y} ||^2 = \big(\mathbf{X} \beta - \mathbf{y}\big)^T \big(\mathbf{X} \beta - \mathbf{y}\big) = \beta^T \mathbf{X}^T \mathbf{X} \beta - 2 \mathbf{y}^T \mathbf{X} \beta + \mathbf{y}^T \mathbf{y}.$$

By solving $\partial \mathcal{F}/\partial \beta = 0$. The solution is 

$$\beta = \big( \mathbf{X}^T \mathbf{X} \big)^{-1} \mathbf{X}^T  \mathbf{y}.$$

- The cell below directly calculate the values of $\beta$. 

In [ ]:
# direct calculation
X1 = x.reshape(-1,1)
X0 = np.ones((len(x),1))
X = np.hstack((X1, X0))
Y = y.reshape(-1,1)

beta = np.linalg.inv(X.T @ X) @ X.T @ Y

slope_cal = beta[0, 0]
intercept_cal = beta[1, 0]

print("slope =", slope_cal)
print("intercept =", intercept_cal)

- The cell below uses the built-in function `polyfit`.

In [ ]:
# np.polyfit
p = np.polyfit(x, y, 1)

slope_poly = p[0]
intercept_poly = p[1]

print("polyfit slope =", p[0])
print("polyfit intercept =", p[1])

### 1.2 `OLS` ordinary least square fitting.

Below, we use `OLS` function in `statsmodels` library to fit the data.
- import the library
- add constant. This makes the design matrix:

$$\mathbf{X} = \begin{bmatrix} 1 & x_1 \\ 1 & x_2 \\ . \\ . \\ 1 & x_n \end{bmatrix}$$

- extract slope and intercept.


In [ ]:
# statsmodel
import statsmodels.api as sm

# x and y from your example
X = sm.add_constant(x)   # adds intercept column

# initialize a model_OLS object
model_OLS = sm.OLS(y, X)
results_OLS = model_OLS.fit()

print(results_OLS.summary())

&#9989; Do this: What are R$^2$ value and $p$ value in statistics?

In [ ]:
intercept_OLS = results_OLS.params[0]
slope_OLS = results_OLS.params[1]

print("slope =", slope_OLS)
print("intercept =", intercept_OLS)

#### 1.3 `curve_fit` method in `scipy.optimiza` library 

Mathematically, `curve_fit` minimizes:

$$\min_i \big(y-f(x_i, p)\big)^2$$

where: $p$ = fitting parameters. For the linear model:

$$f(x,a,b)=ax+b.$$

Below we 
- define the function to be fitted.
- use `curve_fit`.


In [ ]:
def linear_model(x, a, b):
    return a*x + b

In [ ]:
from scipy.optimize import curve_fit

popt, pcov = curve_fit(linear_model, x, y)

slope_cf = popt[0]
intercept_cf = popt[1]

print("slope =", slope)
print("intercept =", intercept)


In [ ]:
print(pcov)

&#9989; Do this: What is `pcov`? What do values mean?

#### 1.4 Use `least_squares` in `scipy.optimize` library. 

This should be familiar to you because we have use it in finding common tangent lines before.

Mathematically, `least_squares` minimizes:

$$\min_i \big(r_i \big)^2,$$

where:

$$r_i = ax_i+b - y_i.$$

Below, the code
- define the residual.
- find the solution that minimizes the residual.


In [ ]:
def residual(p, x, y):
    a, b = p
    return a*x + b - y

In [ ]:
from scipy.optimize import least_squares

res_ls = least_squares(residual, x0=[1,1], args=(x,y))

slope_ls = res_ls.x[0]
intercept_ls = res_ls.x[1]

print("slope =", slope_ls)
print("intercept =", intercept_ls)

**Now, let's plot the results from all these methods.**

In [ ]:
# visualization
# evaluate fitted polynomial
y_polyfit = np.polyval(p, x)

# evaluate fitted OLS model
y_OLS = results_OLS.predict(X)

# evaluate fitted curve_fit model
y_cf = linear_model(x, *popt)

# evaluate fitted curve_fit model
y_ls = linear_model(x, slope_ls, intercept_ls)

plt.figure(figsize=(6,4))

plt.scatter(x, y, label='noisy data')
plt.plot(x, y_polyfit, 'r', linewidth=2, label='polyfit')
plt.plot(x, y_OLS, 'g:', linewidth=2, label='OLS')
plt.plot(x, y_cf, 'kx', linewidth=2, label='curve_fit')
plt.plot(x, y_ls, 'yd', linewidth=2, label='least_squares')


plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)
plt.legend()
plt.show()

Note that, in `curve_fit`, we define the function, but in `least_squares`, we define the residual. Note that `least_squares` is more flexible. Recall the common tangent problem, `least_squares` finds the roots for multiple equations.

---
### More practices on regression.



Here, let's practice these methods for fitting a 4th order function. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# reproducibility
np.random.seed(0)

# x values
x = np.linspace(0, 1, 50)

# true line
y_true = ((x-0.2)**2 * (0.9-x)**2)*1 + (0.03*x + 0.2)*1

# add random noise
noise = np.random.normal(loc=0, scale=0.003, size=len(x))

y = y_true + noise

plt.figure(figsize=(6,4))

plt.scatter(x, y)

plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)
plt.show()

For a 4th-order polynomial, you will get coefficients for `np.polyfit`:

$$ax^4+bx^3+cx^2+dx+e.$$

In [ ]:
# polynomial fit
degree = 4

p = np.polyfit(x, y, degree)
print(p)

# evaluate fitted polynomial
y_polyfit = np.polyval(p, x)

- use `curve_fit`.

In [ ]:
from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt

def poly4(x, a, b, c, d, e):
    return a*x**4 + b*x**3 + c*x**2 + d*x + e

popt, pcov = curve_fit(poly4, x, y)

a, b, c, d, e = popt

print("a =", a)
print("b =", b)
print("c =", c)
print("d =", d)
print("e =", e)

y_curvefit = poly4(x, *popt)

- Use `least_squares`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

# residual function
def residual(p, x, y):
    a, b, c, d, e = p
    y_model = poly4(x, a, b, c, d, e)
    return y_model - y

# initial guess
p0 = [1, 1, 1, 1, 1]

# optimization
res = least_squares(residual, p0, args=(x, y))

# fitted coefficients
a, b, c, d, e = res.x

print("a =", a)
print("b =", b)
print("c =", c)
print("d =", d)
print("e =", e)

# fitted curve
y_ls = poly4(x, a, b, c, d, e)


---
#### Visualization

In [ ]:
plt.figure(figsize=(7,5))

plt.scatter(x, y, label='noisy data')

plt.plot(x, y_true, 'r', linewidth=2, label='true function')
plt.plot(x, y_polyfit, 'g:', linewidth=2, label='polyfit function')
plt.plot(x, y_curvefit, 'kx', linewidth=2, label='polyfit function')
plt.plot(x, y_ls, 'yd', linewidth=2, label='least_squares fit')

plt.xlabel("x")
plt.ylabel("y")

plt.legend()

plt.show()

&#9989; Do this: Why do the fitted results deviate from the true function? 

---
**This table summarizes these regression methods.**

| Feature | `np.polyfit` | `statsmodels.OLS` | `scipy.optimize.curve_fit` | `scipy.optimize.least_squares` |
|---|---|---|---|---|
| Main purpose | Polynomial fitting | Statistical linear regression | Nonlinear curve fitting | General residual minimization |
| Typical use | Quick polynomial regression | Statistical analysis | Scientific model fitting | Inverse problems / optimization |
| Model type | Polynomial only | Linear in coefficients | Arbitrary nonlinear function | Arbitrary residual equations |
| User defines model? | No | Indirectly via design matrix | Yes | Yes |
| User defines residuals? | No | No | No | Yes |
| Handles nonlinear models | Limited | Mostly linear | Yes | Yes |
| Statistical analysis | Minimal | Excellent | Moderate | Minimal |
| Confidence intervals | Limited | Yes | Approximate via covariance | Not automatic |
| p-values | No | Yes | No | No |
| R² reporting | No | Yes | No | No |
| Covariance matrix | Optional | Yes | Yes (`pcov`) | Not automatic |
| Optimization method | Linear least squares | Linear least squares | Nonlinear least squares | General nonlinear least squares |
| Requires initial guess | No | No | Sometimes | Usually yes |
| Supports constraints/bounds | No | Limited | Limited | Excellent |
| Ease of use | Very easy | Easy | Moderate | More advanced |
| Flexibility | Low | Moderate | High | Very high |
| Numerical stability | Good for low degree | Good | Good | Excellent control |
| Good for teaching? | Excellent intro | Excellent statistics | Excellent nonlinear fitting | Excellent optimization/inverse problems |
| Common scientific uses | Polynomial trends | Regression analysis | Exponential/Gaussian fitting | Constitutive laws, PDE inverse problems |
| Typical syntax | `np.polyfit(x,y,n)` | `sm.OLS(y,X)` | `curve_fit(f,x,y)` | `least_squares(residual)` |
| Strength | Simplicity | Statistical interpretation | Easy nonlinear fitting | Maximum flexibility |
| Weakness | Polynomial only | Mostly linear models | Less flexible than LSQ | More complicated |

# You're done with this file. Please upload the complete notebook file to the [**Link**](https://www.dropbox.com/request/pdgeetejng1pnkp8jr70)